# Student Notebook - MCP + Airbnb (Colab)

Reference notebook: local notes MCP + Airbnb MCP + optional real LLM.

## Install
Run once. npm only needed for the real Airbnb server.

In [1]:
%pip install -q "mcp[cli]>=1.0,<2.0" nest_asyncio
# Optional real LLM: %pip install -q azure-ai-inference
# Optional real Airbnb server (requires npm): !npm install -g @openbnb/mcp-server-airbnb


In [2]:
# MCP subprocess stderr is redirected to subprocess.DEVNULL below.
# No monkey-patching of Jupyter/Colab stdout or stderr is necessary.
import subprocess


## Config
Flip toggles as needed. Keep defaults for stubbed run.

In [3]:
import os
from pathlib import Path

MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "")  # Not required in STDIO stub mode
USE_REAL_AIRBNB = False
USE_REAL_LLM = False  # No GITHUB_TOKEN needed in stub mode


In [4]:
BASE_ENV = os.environ.copy()
if MCP_HTTP_TOKEN:
    BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN


In [5]:
# Optional real LLM only: configure GITHUB_TOKEN in Colab Secrets.
# Do not print or hardcode tokens.
if USE_REAL_LLM:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Configure GITHUB_TOKEN in Colab Secrets.")
    os.environ["GITHUB_TOKEN"] = token


## Local notes MCP server

In [6]:
LOCAL_SERVER = Path("local_notes_server.py")
LOCAL_SERVER.write_text(
    """from mcp.server.fastmcp import FastMCP

mcp = FastMCP("NotesDemo")
notes = []

@mcp.tool()
def add_note(text: str) -> str:
    \"\"\"Add a note to the in-memory list.\"\"\"
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"

@mcp.tool()
def list_notes() -> str:
    \"\"\"List notes saved in this server process.\"\"\"
    if not notes:
        return "No notes yet"
    return "\\n".join(f"{i}. {note}" for i, note in enumerate(notes, 1))

if __name__ == "__main__":
    mcp.run(transport="stdio")
""",
    encoding="utf-8",
)
print("wrote", LOCAL_SERVER)


wrote local_notes_server.py


In [7]:
AIRBNB_STUB = Path("airbnb_stub_server.py")
AIRBNB_STUB.write_text(
    """from mcp.server.fastmcp import FastMCP

mcp = FastMCP("AirbnbStub")
LISTINGS = {
    "paris": [
        {"title": "Central Paris Studio", "price_per_night_usd": 110},
        {"title": "Montmartre Apartment", "price_per_night_usd": 145},
    ],
    "london": [{"title": "London City Flat", "price_per_night_usd": 160}],
    "new york": [{"title": "Manhattan Room", "price_per_night_usd": 190}],
}

@mcp.tool()
def search_listings(city: str) -> dict:
    \"\"\"Return illustrative static Airbnb listings for a city (not live availability).\"\"\"
    key = city.strip().lower()
    return {"city": city.strip(), "listings": LISTINGS.get(key, []), "stub": True}

if __name__ == "__main__":
    mcp.run(transport="stdio")
""",
    encoding="utf-8",
)
print("wrote", AIRBNB_STUB)


wrote airbnb_stub_server.py


## Client helpers (convert tools, stub planner, optional real LLM)

In [8]:
import asyncio
import json
import re
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def convert_tool(tool, prefix: str) -> dict:
    """Convert an MCP tool to a function specification for an LLM."""
    schema = tool.inputSchema or {}
    return {
        "type": "function",
        "function": {
            "name": f"{prefix}__{tool.name}",
            "description": tool.description or "MCP tool",
            "parameters": {
                "type": "object",
                "properties": schema.get("properties", {}),
                "required": schema.get("required", []),
            },
        },
    }

def stub_plan(prompt: str, functions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Plan a listing search and a note action without external model calls."""
    available = {f["function"]["name"] for f in functions}
    city_match = re.search(r"\\b(paris|london|new york)\\b", prompt, re.I)
    city = city_match.group(0) if city_match else "Paris"
    calls = []
    if "airbnb__search_listings" in available:
        calls.append({"name": "airbnb__search_listings", "args": {"city": city}})
    note_match = re.search(r"(?:save|add|create|write|salve|adicione|crie)\\s+(?:a\\s+)?note\\s*[:\\-]?\\s*(.+)", prompt, re.I)
    note_text = note_match.group(1).strip() if note_match else f"Review Airbnb options in {city}"
    if "notes__add_note" in available:
        calls.append({"name": "notes__add_note", "args": {"text": note_text}})
    if "notes__list_notes" in available:
        calls.append({"name": "notes__list_notes", "args": {}})
    if not calls:
        print("Warning: no supported tools found for the stub planner.")
    return calls

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    """Return tool calls from the stub planner or optional GitHub Models."""
    if not use_real:
        return stub_plan(prompt, functions)
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use the stub planner.")
    client = ChatCompletionsClient(
        endpoint="https://models.inference.ai.azure.com",
        credential=AzureKeyCredential(token),
    )
    resp = client.complete(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "Select MCP functions to fulfill the user request. Use tools when appropriate."},
            {"role": "user", "content": prompt},
        ],
        tools=functions,
        tool_choice="auto",
        temperature=0,
    )
    calls = []
    for tc in resp.choices[0].message.tool_calls or []:
        args = tc.function.arguments
        calls.append({
            "name": tc.function.name,
            "args": json.loads(args) if isinstance(args, str) else args,
        })
    return calls


In [9]:
def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = True,
) -> str:
    import os
    import json

    # MINIMAL FIX: shrink tool_results before sending to gpt-4o
    small_results = []
    for r in tool_results:
        content = r.get("content", [])
        short_content = []
        if content:
            first = content[0]
            if isinstance(first, str) and len(first) > 4000:
                first = first[:4000] + "...(truncated)..."
            short_content = [first]
        small_results.append(
            {
                "name": r.get("name"),
                "args": r.get("args", {}),
                "content": short_content,
            }
        )


    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use_real=False in answer_with_llm.")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        # use the shrunk version here
        "tool_results": small_results,
    }

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer the user's question using the given tool outputs.\n"
                    "JSON contains user_question, tool_calls, and tool_results (already truncated).\n"
                    "1. Answer clearly in markdown.\n"
                    "2. At the end, add:\n"
                    "## Tools used\n"
                    "- One bullet per distinct tool name.\n"
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    msg = resp.choices[0].message
    parts = getattr(msg, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            text = getattr(p, "text", None) or getattr(p, "content", None)
            if isinstance(text, str):
                texts.append(text)
        if texts:
            return "".join(texts)

    return str(msg.content)


## Orchestrate (connect both servers and execute tool_calls)

In [10]:
async def orchestrate(prompt: str):
    """Connect both MCP servers, discover tools, plan calls and execute them."""
    local_params = StdioServerParameters(
        command="mcp", args=["run", str(LOCAL_SERVER)], env=BASE_ENV
    )
    if USE_REAL_AIRBNB:
        airbnb_params = StdioServerParameters(
            command="npx",
            args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
            env=BASE_ENV,
        )
    else:
        airbnb_params = StdioServerParameters(
            command="mcp", args=["run", str(AIRBNB_STUB)], env=BASE_ENV
        )

    async with stdio_client(local_params, errlog=subprocess.DEVNULL) as (lr, lw):
        async with ClientSession(lr, lw) as local_sess:
            await local_sess.initialize()
            local_tools = (await local_sess.list_tools()).tools
            async with stdio_client(airbnb_params, errlog=subprocess.DEVNULL) as (ar, aw):
                async with ClientSession(ar, aw) as airbnb_sess:
                    await airbnb_sess.initialize()
                    airbnb_tools = (await airbnb_sess.list_tools()).tools
                    print("Notes tools:", [t.name for t in local_tools])
                    print("Airbnb tools:", [t.name for t in airbnb_tools])
                    functions = (
                        [convert_tool(t, "notes") for t in local_tools]
                        + [convert_tool(t, "airbnb") for t in airbnb_tools]
                    )
                    print("Function specs:", json.dumps(functions, indent=2))
                    tool_calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
                    print("tool_calls:", tool_calls)
                    tool_results = []
                    for call in tool_calls:
                        name, args = call["name"], call["args"]
                        prefix, tool_name = name.split("__", 1)
                        if prefix == "notes":
                            session = local_sess
                        elif prefix == "airbnb":
                            session = airbnb_sess
                        else:
                            raise ValueError(f"Unknown tool prefix: {prefix}")
                        res = await session.call_tool(tool_name, arguments=args)
                        content = [c.text for c in res.content if getattr(c, "type", None) == "text"]
                        tool_results.append({
                            "name": name, "args": args, "content": content,
                            "isError": res.isError,
                        })
                    return tool_calls, tool_results


## Demo
Adjust the prompt as you like. Switch `USE_REAL_AIRBNB/USE_REAL_LLM` to true when ready.

In [11]:
prompt = "Find Airbnb listings in Paris and save a note: Compare the Paris listings."
tool_calls, tool_results = await orchestrate(prompt)
print("\\n=== RESULTS ===")
for item in tool_results:
    print(json.dumps(item, indent=2, ensure_ascii=False))
assert any(x["name"].startswith("airbnb__") for x in tool_calls)
assert any(x["name"] == "notes__add_note" for x in tool_calls)
assert all(not x["isError"] for x in tool_results)
print("\\nDaily Challenge completed successfully.")


Notes tools: ['add_note', 'list_notes']
Airbnb tools: ['search_listings']
Function specs: [
  {
    "type": "function",
    "function": {
      "name": "notes__add_note",
      "description": "Add a note to the in-memory list.",
      "parameters": {
        "type": "object",
        "properties": {
          "text": {
            "title": "Text",
            "type": "string"
          }
        },
        "required": [
          "text"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "notes__list_notes",
      "description": "List notes saved in this server process.",
      "parameters": {
        "type": "object",
        "properties": {},
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "airbnb__search_listings",
      "description": "Return illustrative static Airbnb listings for a city (not live availability).",
      "parameters": {
        "type": "object",
        "properties": {
      